## Notebook Description

### Dataset Description
The data includes EV charging reports, Hourly EV charging loads, and idle capacity, for all sessions and users individually along with weather and traffic data of the location.
The datasets are described in the Data In Brief article ["Residential electric vehicle charging datasets from apartment buildings" (2021)](https://www.sciencedirect.com/science/article/pii/S2352340921003899)

### Notebook purpose
- Practice usage of unsupervised learning algorithms:
    - Session-level clustering
    - User-level clustering
    - Anomaly detection

### Workflow
- Environment creation
- Data collection
- Data preparation
- Exploratory Data Analysis
- Feature Engineering
- Model development
- Results

Step 0: Environment Creation

In [2]:
#%pip install numpy; pandas; matplotlib; seaborn; scikit-learn; kagglehub

In [3]:
import numpy as np
import pandas as pd
import sklearn
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns


c:\D_Folder\GitFiles\DataScienceProjects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 1: Data collection

In [4]:
from pathlib import Path

output_dir = Path('datasets')
output_dir.mkdir(exist_ok=True)

path = kagglehub.dataset_download("anshtanwar/residential-ev-chargingfrom-apartment-buildings", output_dir=output_dir)

print("Path to dataset files:", path)

csv_path = output_dir / 'archive'
print("Path to the csv files", csv_path)

Path to dataset files: datasets
Path to the csv files datasets\archive


#### Dataset 1: EV Charging Reports
The CSV file “Dataset 1” describes 6,878 individual charging sessions, registered by 97 user IDs from December 2018 to January 2020.

<details>
<summary>Dataset 1: EV charging reports, describing each individual EV charging session.</summary>

| Column | Description |
|---|---|
| `session_ID` | Unique ID for EV charging session (N=6878) |
| `Garage_ID` | ID for garage address (N=24) |
| `User_ID` | ID for user (N=97) |
| `User_type` | Charging Port (CP) ownership: private or shared CPs |
| `Shared_ID` | ID for shared CP when used (N=12) |
| `Start_plugin` | Plug-in date and time (format: 21.12.2018 10:20) |
| `Start_plugin_hour` | Clock hour for plug-in (00–23) |
| `End_plugout` | Plug-out date and time (format: 21.12.2018 10:20) |
| `End_charging_hour` | Clock hour for plug-out (00–23) |
| `El_kWh` | Charged energy (kWh) |

</details>

In [ ]:
df1 = pd.read_csv(csv_path / 'Dataset 1_EV charging reports.csv', sep=';', decimal=',')

df1 = df1.dropna(axis=0, subset=['End_plugout']) #Removed na because it meant the plug-out time is to early compared to energy charged

float_columns = ['Start_plugin_hour', 'End_plugout_hour', 'El_kWh', 'Duration_hours']
timestamp_columns = ['Start_plugin', 'End_plugout']
df1[float_columns] = df1[float_columns].astype(float)
df1[timestamp_columns] = df1[timestamp_columns].apply(
    lambda column: pd.to_datetime(column, format="%d.%m.%Y %H:%M")
)
df1['Shared_ID'] = df1['Shared_ID'].fillna(0)
df1['Shared_ID'] = df1['Shared_ID'].astype('str')

df1.info()

   session_ID Garage_ID User_ID User_type Shared_ID      Start_plugin  \
0           1      AdO3  AdO3-4   Private       NaN  21.12.2018 10:20   
1           2      AdO3  AdO3-4   Private       NaN  21.12.2018 10:24   
2           3      AdO3  AdO3-4   Private       NaN  21.12.2018 11:33   
3           4      AdO3  AdO3-2   Private       NaN  22.12.2018 16:15   
4           5      AdO3  AdO3-2   Private       NaN  24.12.2018 22:03   

   Start_plugin_hour       End_plugout  End_plugout_hour  El_kWh  \
0                 10  21.12.2018 10:23              10.0    0.30   
1                 10  21.12.2018 10:32              10.0    0.87   
2                 11  21.12.2018 19:46              19.0   29.87   
3                 16  23.12.2018 16:40              16.0   15.56   
4                 22  24.12.2018 23:02              23.0    3.62   

   Duration_hours month_plugin weekdays_plugin             Plugin_category  \
0        0.050000          Dec          Friday         late morning (9-12)

#### Conclusions
- Dropped rows with missing End_plugout values since it meant faulty measurements
- Transformed dtype of columns in float and datetime where needed
- filled missing Share_ID(private chg) with 0

#### Dataset 2: Hourly EV charging loads and idle capacity, for all sessions and users individually
The CSV file “Dataset 2” describes EV charging loads and non-charging idle capacity for each user and all EV charging sessions individually.

<details>
<summary>Dataset 2: Hourly EV charging loads and idle capacity, for all users individually.</summary>

| Column | Description |
|---|---|
| `date_from` | Starting time (format: 22.01.2019 19:00) |
| `date_to` | Ending time (format: 22.01.2019 20:00) |
| `User_ID` | ID for user (N=97) |
| `session_ID` | Unique ID for EV charging session (N=6878) |
| `Synthetic_3_6kW` | Synthetic hourly energy load (kWh/h) assuming 3.6 kW charging power (ref. [1]), for users individually |
| `Synthetic_7_2kW` | Synthetic hourly energy load (kWh/h) assuming 7.2 kW charging power (ref. [1]), for users individually |
| `Flex_3_6kW` | Synthetic hourly idle capacity (kWh/h) assuming 3.6 kW charging power, for users individually |
| `Flex_7_2kW` | Synthetic hourly idle capacity (kWh/h) assuming 7.2 kW charging power, for users individually |

</details>

In [39]:
df2 = pd.read_csv(csv_path / 'Dataset 2_Hourly EV loads - Per user.csv', sep=';', decimal=',')
# print(df2.head(15))
# df2.info()

valid_session = df2['session_ID'].notna()
flex_columns = ['Flex_3_6kW', 'Flex_7_2kW']
df2.loc[valid_session, flex_columns] = df2.loc[valid_session, flex_columns].fillna(0)
df2.tail(20)

timestamp_columns = ['date_from', 'date_to']
df2[timestamp_columns] = df2[timestamp_columns].apply(
    lambda column: pd.to_datetime(column, format="%d.%m.%Y %H:%M")
)

# df2_example = df2[df2['session_ID'] == 6862.0]
# df2_example

df2 = df2.fillna(0)
df2['User_ID'] = df2['User_ID'].astype('str')
df2.info()


<class 'pandas.DataFrame'>
RangeIndex: 88156 entries, 0 to 88155
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date_from        88156 non-null  datetime64[us]
 1   date_to          88156 non-null  datetime64[us]
 2   User_ID          88156 non-null  str           
 3   session_ID       88156 non-null  float64       
 4   Synthetic_3_6kW  88156 non-null  float64       
 5   Synthetic_7_2kW  88156 non-null  float64       
 6   Flex_3_6kW       88156 non-null  float64       
 7   Flex_7_2kW       88156 non-null  float64       
dtypes: datetime64[us](2), float64(5), str(1)
memory usage: 5.4 MB


#### Conclusions
- There are NaN values on:
    - UserID, sessionID - it means that at that hour there is no charging happening
    - Synthetic columns - it means that the car is still connected to the charging port but finished charging(0 kW)
    - Flex columns - it means that at that hour there is no charging happening - no idle chg capacity available
- Based on these obervations:
    - Synthetic columns will be filled with 0
    - No charging rows will befilled with 0(this can change based on further processing)

#### Dataset 3: Hourly EV charging loads and idle capacity, aggregated for private or shared CPs
The CSV files “Dataset 3a” and “Dataset 3b” describe EV charging loads and idle capacity, aggregated for users with private (3a) or shared (3b) CPs. Charging power 3.6 kW or 7.2 kW is assumed, with immediate charging after plug-in. 

<details>
<summary>Dataset 3: Hourly EV charging loads and idle capacity, aggregated for private or shared CPs</summary>

| Column | Description |
|---|---|
| `date_from` | Starting time (format: 22.01.2019 19:00) |
| `daily_hour` | Clock hour (00–23) |
| `weekday` | Weekday (Monday–Sunday) |
| `month` | Month (January–December) |
| `Synthetic_3_6kW` | Synthetic hourly energy load (kWh/h) assuming 3.6 kW charging power, aggregated for users with private (2a) or shared (2b) CPs |
| `Synthetic_7_2kW` | Synthetic hourly energy load (kWh/h) assuming 7.2 kW charging power, aggregated for users with private (2a) or shared (2b) CPs |
| `Flex_3_6kW` | Synthetic hourly idle capacity (kWh/h) assuming 3.6 kW charging power, aggregated for users with private (2a) or shared (2b) CPs |
| `Flex_7_2kW` | Synthetic hourly idle capacity (kWh/h) assuming 7.2 kW charging power, aggregated for users with private (2a) or shared (2b) CPs |
| `2a: n_private` | Number of registered User IDs using private CPs (increasing, 1 to 58) |
| `2b: n_shared` | Number of registered User IDs using shared CPs (increasing, 1 to 24) |

</details>

In [40]:
df3a = pd.read_csv(csv_path / 'Dataset 3a_Hourly EV loads - Aggregated private.csv', sep=';', decimal=',')

df3a['date_from'] = pd.to_datetime(df3a['date_from'], format="%d.%m.%Y %H:%M")

print(df3a.head(1))
print(df3a.tail(1))
df3a.info()

            date_from  daily_hour weekday month  Synthetic_3_6kW  \
0 2018-12-21 10:00:00          10  Friday   Dec             1.17   

   Synthetic_7_2kW  Flex_3_6kW  Flex_7_2kW  n_private  
0             1.17         0.0       0.174          1  
               date_from  daily_hour weekday month  Synthetic_3_6kW  \
9756 2020-01-31 22:00:00          22  Friday   Jan              0.0   

      Synthetic_7_2kW  Flex_3_6kW  Flex_7_2kW  n_private  
9756              0.0       1.728       3.456         58  
<class 'pandas.DataFrame'>
RangeIndex: 9757 entries, 0 to 9756
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date_from        9757 non-null   datetime64[us]
 1   daily_hour       9757 non-null   int64         
 2   weekday          9757 non-null   str           
 3   month            9757 non-null   str           
 4   Synthetic_3_6kW  9757 non-null   float64       
 5   Synthetic_7_2kW  97

In [41]:
df3b = pd.read_csv(csv_path / 'Dataset 3b_Hourly EV loads - Aggregated shared.csv', sep=';', decimal=',')

df3b['date_from'] = pd.to_datetime(df3b['date_from'], format="%d.%m.%Y %H:%M")

print(df3b.head())
df3b.info()

            date_from  daily_hour   weekday month  Synthetic_3_6kW  \
0 2019-01-10 00:00:00           0  Thursday   Jan              0.0   
1 2019-01-10 01:00:00           1  Thursday   Jan              0.0   
2 2019-01-10 02:00:00           2  Thursday   Jan              0.0   
3 2019-01-10 03:00:00           3  Thursday   Jan              0.0   
4 2019-01-10 04:00:00           4  Thursday   Jan              0.0   

   Synthetic_7_2kW  Flex_3_6kW  Flex_7_2kW  n_shared  
0              0.0         0.0         0.0         1  
1              0.0         0.0         0.0         1  
2              0.0         0.0         0.0         1  
3              0.0         0.0         0.0         1  
4              0.0         0.0         0.0         1  
<class 'pandas.DataFrame'>
RangeIndex: 9288 entries, 0 to 9287
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date_from        9288 non-null   datetime64

##### Conclusions:
- No missing data in the 3a & 3b columns
- Synthetic+Flex columns that were NaN in df2 are 0 in this dfs
- 3b df starts 20 days later -> less entries
- Later start happens simply because the first shared session starts 20 days later


#### Dataset 4  Dataset 4: Average EV charging loads per user, for each daily hour during weekdays/Saturdays/Sundays 

<details>
<summary>Dataset description</summary>

Dataset 4 in Table 4 shows average EV charging loads per user, for each daily hour during weekdays, Saturdays, and Sundays. Charging power 7.2 kW is assumed, with immediate charg- ing after plug-in. In the table, charging loads for users with private and shared CPs are shown separately. The daily charging load profiles are based on the period with 30 to 82 users, from June 2019 to January 2020, with the number of users with private CPs increasing from 18 to 58, and users with shared CPs increasing from 12 to 24. The subset of the period is chosen, to get a more representative overview of expected power per user for aggregated loads. 

</details>

In [42]:
df4 = pd.read_csv(csv_path / 'Dataset 4_EV_Charging_Loads.csv')

df4.head()
df4.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   daily_hour         24 non-null     str    
 1   private_weekdays   24 non-null     float64
 2   private_saturdays  24 non-null     float64
 3   private_sundays    24 non-null     float64
 4   shared_weekdays    24 non-null     float64
 5   shared_saturdays   24 non-null     float64
 6   shared_sundays     24 non-null     float64
dtypes: float64(6), str(1)
memory usage: 1.4 KB


#### Conclusions:
- Dataset is complete, no processing needed

#### Dataset 5: Hourly smart meter data from garage Bl2.
The CSV file “Dataset 5” describes hourly smart meter data from garage Bl2, with aggregated electricity use each hour. The dataset also includes synthetic hourly energy loads, aggregated for the same garage.

<details>
<summary>Column descriptions</summary>

| Column | Description |
|---|---|
| `Garage_ID` | ID for garage address (Bl2) |
| `date_from` | Starting time (format: 22.01.2019 19:00) |
| `date_to` | Ending time (format: 22.01.2019 20:00) |
| `month` | Measurement starting month |
| `AMS_kWh` | Aggregated electricity use in the garage each hour, measured by AMS meter |
| `Synthetic_3_6kW` | Synthetic hourly energy load (kWh/h) assuming 3.6 kW charging power, aggregated for users in the garage |
| `Synthetic_7_2kW` | Synthetic hourly energy load (kWh/h) assuming 7.2 kW charging power, aggregated for users in the garage |
| `Simultaneous_if_3_6kW` | Number of simultaneous charging sessions, assuming all sessions charge with 3.6 kW charging power. NA if no charging sessions are assumed |

</details>

In [43]:
df5 = pd.read_csv(csv_path / 'Dataset 5_AMS data from garage Bl2.csv', sep=';', decimal=',')
#print(df5[df5['date_to'].isna()])
#print(df5.iloc[1500:1510])

timestamp_columns = ['date_from', 'date_to']
df5[timestamp_columns] = df5[timestamp_columns].apply(
    lambda column: pd.to_datetime(column, format="%d.%m.%Y %H:%M")
)

df5.loc[1506, 'date_to'] = '06.03.2019 14:00'
df5.loc[1507, 'date_to'] = '06.03.2019 15:00'

df5.iloc[1500:1515]
df5 = df5.fillna(0)
df5.info()

<class 'pandas.DataFrame'>
RangeIndex: 9466 entries, 0 to 9465
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date_from              9466 non-null   datetime64[us]
 1   date_to                9466 non-null   datetime64[us]
 2   month                  9466 non-null   str           
 3   AMS_kWh                9466 non-null   float64       
 4   Synthetic_3_6kW        9466 non-null   float64       
 5   Synthetic_7_2kW        9466 non-null   float64       
 6   Simultaneous_if_3_6kW  9466 non-null   float64       
dtypes: datetime64[us](2), float64(4), str(1)
memory usage: 517.8 KB


#### Conclusions
- There are missing values in 06.03.2019 at 13:00; 14:00 probably due to an sensor error
- Energy and power for these rows will be set to 0 
- Energy and power for all rows with NA are set to 0 since NA was used if no charging session assumed

#### Dataset 6: Local traffic density
The CSV file “Dataset 6” describes local hourly traffic density in 5 nearby traffic locations, downloaded from [3]. The data includes an hourly count of vehicles shorter than 5.6 meter

<details>
<summary>Dataset 6: Local traffic density </summary>

| Column | Description |
|---|---|
| `Date_from` | Starting time (format: 22.01.2019 19:00) |
| `Date_to` | Ending time (format: 22.01.2019 20:00) |
| `Location 1 to 5` | Number of vehicles shorter than 5.6 meters each hour at five nearby traffic locations |

</details>

In [44]:
df6 = pd.read_csv(csv_path / 'Dataset 6_Local traffic distribution.csv', sep=';', decimal=',')
df6.info()
df6.head()

timestamp_columns = ['Date_from', 'Date_to']
df6[timestamp_columns] = df6[timestamp_columns].apply(
    lambda column: pd.to_datetime(column, format="%d.%m.%Y %H:%M")
)

locations = ['KROPPAN BRU', 'MOHOLTLIA', 'SELSBAKK', 'MOHOLT RAMPE 2', 'Jonsvannsveien vest for Steinanvegen']
datetimes = ['Date_from', 'Date_to']

df6[locations] = df6[locations].replace('-', '0')

df6[locations] = df6[locations].astype(int)
df6[datetimes] = df6[datetimes].apply(
    lambda datetime: pd.to_datetime(datetime, format='%d.%m.%Y %H:%M')
)

df6.info()

<class 'pandas.DataFrame'>
RangeIndex: 10248 entries, 0 to 10247
Data columns (total 7 columns):
 #   Column                                Non-Null Count  Dtype
---  ------                                --------------  -----
 0   Date_from                             10248 non-null  str  
 1   Date_to                               10248 non-null  str  
 2   KROPPAN BRU                           10248 non-null  str  
 3   MOHOLTLIA                             10248 non-null  str  
 4   SELSBAKK                              10248 non-null  str  
 5   MOHOLT RAMPE 2                        10248 non-null  int64
 6   Jonsvannsveien vest for Steinanvegen  10248 non-null  int64
dtypes: int64(2), str(5)
memory usage: 560.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 10248 entries, 0 to 10247
Data columns (total 7 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Date_from           

##### Conclusions
- No missing data in this dataframe
- Converted Date_from and Date_to columns to dataframe
- Converted the rest of the columns from string to int

#### Dataset Weather: Weather for Trondheim

<details>
<summary>Dataset Weather: Weather for Trondheim</summary>

| Column | Description |
|---|---|
| `Unnamed: 0` | Row index included in the CSV; likely not a weather measurement |
| `clouds` | Cloud cover |
| `datetime` | Date of the weather record |
| `dewpt` | Dew-point temperature |
| `dhi` | Diffuse horizontal solar irradiance |
| `dni` | Direct normal solar irradiance |
| `ghi` | Global horizontal solar irradiance |
| `max_dhi` | Maximum diffuse horizontal solar irradiance |
| `max_dni` | Maximum direct normal solar irradiance |
| `max_ghi` | Maximum global horizontal solar irradiance |
| `max_temp` | Maximum temperature |
| `max_temp_ts` | Timestamp for the maximum temperature |
| `max_uv` | Maximum UV index |
| `max_wind_dir` | Wind direction when wind speed was at its maximum |
| `max_wind_spd` | Maximum wind speed |
| `max_wind_spd_ts` | Timestamp for the maximum wind speed |
| `min_temp` | Minimum temperature |
| `min_temp_ts` | Timestamp for the minimum temperature |
| `precip` | Precipitation |
| `precip_gpm` | Precipitation measurement reported as `precip_gpm` in the source data |
| `pres` | Atmospheric pressure |
| `revision_status` | Weather-data revision status |
| `rh` | Relative humidity |
| `slp` | Sea-level pressure |
| `snow` | Snowfall |
| `snow_depth` | Snow depth |
| `solar_rad` | Solar radiation |
| `t_dhi` | Total diffuse horizontal solar irradiance |
| `t_dni` | Total direct normal solar irradiance |
| `t_ghi` | Total global horizontal solar irradiance |
| `t_solar_rad` | Total solar radiation |
| `temp` | Temperature |
| `ts` | Timestamp for the weather record |
| `wind_dir` | Wind direction |
| `wind_gust_spd` | Wind gust speed |
| `wind_spd` | Wind speed |
</details>

In [ ]:
df_weather = pd.read_csv(csv_path / 'Norway_Trondheim_ExactLoc_Weather.csv')
print(df_weather[['snow', 'snow_depth']].head(5))
df_weather = df_weather.drop(columns=['Unnamed: 0', 'snow_depth'], errors='ignore')

df_weather['datetime'] = pd.to_datetime(df_weather['datetime'], errors='coerce')
print(df_weather['datetime'].head())
df_weather.info()

   Unnamed: 0  clouds    datetime  dewpt  dhi  dni  ghi  max_dhi  max_dni  \
0           0      66  2018-12-01   -1.7    5   40    6       34      288   
1           1      64  2018-12-02   -2.8    5   38    6       33      281   
2           2     100  2018-12-03   -0.7    5   37    5       33      273   
3           3      93  2018-12-04   -0.5    5   35    5       32      266   
4           4      87  2018-12-05   -1.9    5   34    5       32      260   

   max_ghi  ...  solar_rad  t_dhi  t_dni  t_ghi  t_solar_rad  temp  \
0       48  ...          2    127    966    144           52   7.4   
1       46  ...          5    123    923    135          126   5.4   
2       44  ...          1    119    881    128           21   2.4   
3       42  ...          1    115    842    121           22   0.8   
4       41  ...          2    111    804    114           39  -0.4   

           ts  wind_dir  wind_gust_spd  wind_spd  
0  1543618800       187           12.8       5.5  
1  1543705200 

##### Conclusions
- No missing data apart from snow snow depth feature
- Dropped Unnamed: 0, snow_depth(all nan values and redundant with snow) columns
- Converted datetime column from string to datetime